<a href="https://colab.research.google.com/github/sejuti009/Welcome-To-Data-Science-/blob/main/RecommendationEngineModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# import libraries
import pandas as pd
import numpy as np

In [4]:
# load data
rating_df = pd.read_csv('/content/ratings.csv')
movies_df = pd.read_csv('/content/movies.csv')

In [5]:
# data processing
# Extract year
movies_df['year'] = movies_df['title'].str.extract(r'\((\d{4})\)')
# Remove year from title
movies_df['title'] = movies_df['title'].str.replace(r'\(\d{4}\)', '', regex=True).str.strip()
# Split genres
movies_df['genres'] = movies_df['genres'].str.split('|')

In [6]:
# one-hot encoding using explode (FAST)
movies_exploded = movies_df.explode('genres')

In [7]:
movies_onehot = pd.crosstab(movies_exploded['movieId'],movies_exploded['genres'])
# here crosstab function is used to create the contigency table

In [8]:
# merge back
movies_copy = movies_df.merge(movies_onehot,on='movieId')

In [9]:
# drop timestamp
rating_df.drop('timestamp',axis=1,inplace=True)

In [10]:
# user input
"""user_input = [
    {'title': 'Scream', 'rating': 5},
    {'title': 'The Shining', 'rating': 5},
    {'title': 'Halloween', 'rating': 4.5},
    {'title': 'Nightmare on Elm Street', 'rating': 5},
    {'title': 'The Exorcist', 'rating': 4.5},
    {'title': 'Jumanji', 'rating': 2} # dislke non-horror
]"""

user_input = [
{'title': 'Toy Story', 'rating': 5},
{'title': 'Finding Nemo', 'rating': 5},
{'title': 'Frozen', 'rating': 4.5},
{'title': 'The Lion King', 'rating': 5},
{'title': 'Moana', 'rating': 4.5},
{'title': 'The Conjuring', 'rating': 1}
]

In [11]:
movies_input = pd.DataFrame(user_input)

In [12]:
# match title with dataset
input_movies = movies_df[movies_df['title'].isin(movies_input['title'])]

In [13]:
# merge to get movieId
movies_input = pd.merge(input_movies,movies_input,on='title')

In [14]:
# if no matches found
if movies_input.empty:
  raise ValueError('No matching movies found. Check titles!')

In [15]:
# user profile
# get one-hot rows for input movies
user_movies = movies_copy[movies_copy['movieId'].isin(movies_input['movieId'])]

In [16]:
# reset indew
user_movies = user_movies.reset_index(drop=True)
movies_input = movies_input.reset_index(drop=True)

In [17]:
# create genre matrix
user_genres = user_movies.drop(columns=['movieId','title','genres','year'])

In [18]:
# compute user profile
user_profile = user_genres.T.dot(movies_input['rating'])

In [19]:
# recommendation
# full gerne table
genre_table = movies_copy.set_index('movieId')
genre_table = genre_table.drop(columns=['title','genres','year'])

In [20]:
# compute scores
recommendation_scores = genre_table.dot(user_profile)

In [21]:
# normalize (avoid divide by zeros)
# this works if user gives all the ratings of the movies as 0
if user_profile.sum() != 0:
  recommendation_scores = recommendation_scores / user_profile.sum()

In [22]:
# sort recommendation
recommendation_scores = recommendation_scores.sort_values(ascending=False)

In [23]:
# top 20 movies IDs
top_movies = recommendation_scores.head(20).index

In [24]:
# final recommendation table
recommendation_table = movies_df[movies_df['movieId'].isin(top_movies)]

In [25]:
print(recommendation_table[['title','year']])

                                                  title  year
0                                             Toy Story  1995
1706                                               Antz  1998
2250                           Who Framed Roger Rabbit?  1988
2355                                        Toy Story 2  1999
2809            Adventures of Rocky and Bullwinkle, The  2000
3194                                              Shrek  2001
3568                                     Monsters, Inc.  2001
6194                                          Wild, The  2006
6486                                    Shrek the Third  2007
6626                                          Enchanted  2007
6948                            Tale of Despereaux, The  2008
7355                                        Toy Story 3  2010
7360  Shrek Forever After (a.k.a. Shrek: The Final C...  2010
7530                                    Gnomeo & Juliet  2011
7760   Asterix and the Vikings (Astérix et les Vikings)  2006
7805    